In [ ]:
!pip install langchain_community
!pip install langchain_huggingface


In [ ]:
from langchain_huggingface import HuggingFaceEndpoint
repo_id = "mistralai/Mistral-7B-Instruct-v0.3"
HUGGINGFACEHUB_API_TOKEN='' #TOKEN GOES HERE
llm = HuggingFaceEndpoint(
    repo_id=repo_id,
    model_kwargs={"max_length":128},

    temperature=0.5,
    huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN,
)



Langchain

In [ ]:
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
prompt_template = ChatPromptTemplate.from_messages([
     ("system",
"You are a pirate life coach."),
("human",
"How can i get out more and meet new people?"),
("ai","Yarr, board any ship ye see on the sea and expand yar horizons"),
("human","Respond to the question: {question}")
]
)

In [ ]:
llm_chain = prompt_template | llm

In [ ]:
question = "I feel stuck and feel a lack of meaning"


In [ ]:
response = llm_chain.invoke({"question":question})
print(response)

In [ ]:
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory


In [ ]:
memory = ConversationBufferMemory(size=4)
buffer_chain = ConversationChain(llm=llm,memory=memory)

In [ ]:
buffer_chain.invoke("describe time")

In [ ]:
buffer_chain.invoke("describe pine")



---



In [ ]:
from langchain.memory import ConversationSummaryMemory
memory = ConversationSummaryMemory(llm=llm)
summary_chain = ConversationChain(llm=llm, memory=memory, verbose=True)

summary_chain.invoke('what is a lime')
summary_chain.invoke('what is thyme')


In [ ]:
HUGGINGFACEHUB_API_TOKEN = ''

In [44]:
from langchain_huggingface import HuggingFaceEndpoint
from langchain.chains import LLMChain
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate

# Hugging Face model setup
repo_id = "mistralai/Mistral-7B-Instruct-v0.2"
HUGGINGFACEHUB_API_TOKEN = '' #TOKEN GOES HERE
llm = HuggingFaceEndpoint(
    repo_id=repo_id,
    model_kwargs={"max_length":128},
    temperature=0.5,
    huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN,
)

# Create prompt template
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a pirate life coach."),
    ("human", "How can I get out more and meet new people?"),
    ("ai", "Yarr, board any ship ye see on the sea and expand yar horizons."),
    ("human", "Respond to the question: {question}")
])

# Initialize memory
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# Create LLM chain with memory
llm_chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory, verbose=True)


In [ ]:

# Chatbot loop
print("Ahoy! I'm your pirate life coach. Ask me anything, or type 'exit' to quit!")


while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        print("Fair winds and following seas!")
        break

    # Use .invoke() to handle user input
    response = llm_chain.invoke({"question": user_input})

    # Extract and print the AI's response
    ai_response = response.get("text", "Oops, I couldn't come up with a response!")
    print(f"Pirate Coach: {ai_response}")


In [43]:
from langchain_huggingface import HuggingFaceEndpoint
from langchain.chains import LLMChain
from langchain.memory import ConversationSummaryMemory
from langchain_core.prompts import PromptTemplate

# Hugging Face model setup
repo_id = "mistralai/Mistral-7B-Instruct-v0.3"

llm = HuggingFaceEndpoint(
    repo_id=repo_id,
    temperature=0.5,
    model_kwargs={"stop": ["\nHuman:", "Human:", "\nYou:"],"max_length":128},
    huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN,
)

# Create a more controlled prompt template
prompt_template = PromptTemplate(
    input_variables=["history", "human_input"],
    template=(
        "You are a dog accountant who helps people manage their finances creatively. "
        "Provide ONE response to the human's input.\n"
        "DO NOT continue the conversation or generate additional dialogue.\n\n"
        "Previous conversation:\n{history}\n\n"
        "Human: {human_input}\n"
        "Dog Accountant: "  # Remove newline to prevent continuation
    )
)

# Initialize memory with specific summarization template
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="history",
    return_messages=True,
    max_token_limit=500  # Prevent memory from growing too large
)

# Create the chain with specific configuration
llm_chain = LLMChain(
    llm=llm,
    prompt=prompt_template,
    memory=memory,
    verbose=True
)

def get_single_response(chain, user_input):
    """Helper function to ensure we only get one response"""
    response = chain.invoke({"human_input": user_input})
    # Clean up response to ensure we only get the first turn
    ai_text = response["text"].split("\nHuman:")[0].strip()
    return ai_text


In [ ]:

# Chatbot loop
print("Woof! I'm your dog accountant. Ask me anything about managing your finances, or type 'exit' to quit!")

while True:
    user_input = input("\nYou: ").strip()
    if user_input.lower() == "exit":
        print("\nGoodbye! Here's a summary of our conversation:")
        print(memory.load_memory_variables({})["history"])
        break

    try:
        # Get single controlled response
        ai_response = get_single_response(llm_chain, user_input)
        print(f"\nDog Accountant: {ai_response}")

    except Exception as e:
        print(f"\nError: Something went wrong - {str(e)}")
        print("Let's try again!")